# EXE, DIS, and DEC tokenization walkthrough

This notebook trains and applies BPE tokenizers to the `EXE`, `DIS`, and `DEC` artifacts produced by notebook 01. Reusable tokenization functions live in `malweave.tokenization`; this notebook configures the experiment, runs each stage, and makes its data visible.

> `LiftLevel.RAW` means executable-section bytes in the tokenizer API. The `raw/` directory from notebook 01 contains complete PE files; EXE tokenization must read `exe/*.bin` instead.

## 1. Tokenization flow

```text
notebook 01 output
    │
    ├── exe/*.bin ── 16-byte words ── private Unicode mapping ─┐
    ├── dis/*.asm ── one instruction per word ────────────────┼─ BPE training
    └── dec/*.c   ── one code line per word ──────────────────┘
                                                               │
                                                               ├─ tokenizer.json
                                                               ├─ token IDs/sample
                                                               └─ length statistics
```

This flow uses BPE with 16,384 learned tokens. Seven control tokens are added for padding, unknown values, masking, sequence boundaries, classification, and separation. A small corpus may produce fewer learned tokens because BPE can only merge patterns that occur in the training data.

## 2. Environment and package setup

The notebook uses these components from `malweave.tokenization`:

- `LiftLevel` and `TokenizationAlgorithm`
- the seven `SPECIALS` used by the model
- `TokenizationTrainingIterator` and `TrainTokenizer`
- representation-specific normalizer and pre-tokenizer behavior

The package owns decomposition, byte conversion, model selection, and trainer configuration. The notebook supplies local artifact paths and output directories, keeping experiment-specific decisions visible here.

In [ ]:
from collections import OrderedDict
import json
import math
from pathlib import Path
from pprint import pprint
import statistics
import sys
import time

import tokenizers

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from malweave.tokenization import (
    BYTE_TO_UTF8,
    LiftLevel,
    RAW_WORD_SIZE,
    SPECIALS,
    TokenizationAlgorithm,
    TokenizationTrainingIterator,
    TrainTokenizer,
    bytes_to_str_utf8,
)

INPUT_ROOT = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'reproduction'
TOKENIZER_ROOT = PROJECT_ROOT / 'models' / 'tokenizers'
TOKENIZED_ROOT = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'tokenized'
# Maximum number of ordinary BPE tokens; seven special tokens are added separately.
VOCAB_SIZE = 16_384
# Maximum number of artifacts used to learn each representation vocabulary.
TRAIN_FILE_LIMIT = 4_096
# Number of decomposed words yielded to the tokenizer trainer at a time.
BATCH_SIZE = 1_024
# Prevent one learned token from spanning more than 16 mapped bytes/characters.
MAX_TOKEN_LENGTH = 16
print('tokenizers version:', tokenizers.__version__)
print('MalWeave root:', PROJECT_ROOT)
print('Tokenizer package:', PROJECT_ROOT / 'malweave' / 'tokenization')
print('Input root:', INPUT_ROOT)

tokenizers version: 0.20.3
MalWeave root: /Users/jung/Projects/malrec-lab/malweave
Tokenizer package: /Users/jung/Projects/malrec-lab/malweave/malweave/tokenization
Input root: /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/reproduction


## 3. Decomposition and training behavior

`TokenizationTrainingIterator` decomposes EXE into 16-byte words and DIS/DEC into newline-delimited words. `bytes_to_str_utf8` maps each byte `b` to `chr(b + 10752)` so every possible byte has a dedicated Unicode symbol and no byte value is lost.

`TrainTokenizer` configures `models.BPE()`, `BpeTrainer`, the representation-specific normalizer/pre-tokenizer, seven special tokens, requested vocabulary size, and maximum token length. The small check below makes the byte transformation visible without redefining it in the notebook.

In [2]:
sample_bytes = bytes([0x00, 0x41, 0xFF])
mapped = bytes_to_str_utf8(sample_bytes)
print('RAW_WORD_SIZE:', RAW_WORD_SIZE)
print('input bytes:', sample_bytes.hex(' '))
print('mapped code points:', [ord(character) for character in mapped])
print('mapping is exact:', [ord(character) - 10752 for character in mapped] == list(sample_bytes))
print('special tokens:', list(SPECIALS.values()))

RAW_WORD_SIZE: 16
input bytes: 00 41 ff
mapped code points: [10752, 10817, 11007]
mapping is exact: True
special tokens: ['<pad>', '<unk>', '<msk>', '<bos>', '<eos>', '<cls>', '<sep>']


## 4. Inspect tokenizer inputs

The three inputs are matched by SHA-256 filename. This cell reports counts, sizes, and a limited preview before training. It never uses complete PE files from `reproduction/raw/`.

In [3]:
REPRESENTATIONS = OrderedDict({
    'exe': {'lift_level': LiftLevel.RAW, 'files': sorted((INPUT_ROOT / 'exe').glob('*.bin'))},
    'dis': {'lift_level': LiftLevel.DIS, 'files': sorted((INPUT_ROOT / 'dis').glob('*.asm'))},
    'dec': {'lift_level': LiftLevel.DEC, 'files': sorted((INPUT_ROOT / 'dec').glob('*.c'))},
})
assert all(config['files'] for config in REPRESENTATIONS.values()), (
    'Missing EXE/DIS/DEC artifacts. Run notebook 01 with RUN_GHIDRA=True first.'
)

for name, config in REPRESENTATIONS.items():
    files = config['files']
    first = files[0]
    print(f'\n{name.upper()} | lift_level={config["lift_level"].value}')
    print(f'  files={len(files):,}, total_bytes={sum(path.stat().st_size for path in files):,}')
    print(f'  first={first.name}, size={first.stat().st_size:,}')
    if name == 'exe':
        print('  head:', first.read_bytes()[:32].hex(' '))
    else:
        print('  head:')
        print('\n'.join(first.read_text(errors='replace').splitlines()[:5]))


EXE | lift_level=raw
  files=8, total_bytes=1,368,576
  first=d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.bin, size=7,680
  head: 55 8b ec 83 ec 14 0b 05 4c 41 41 00 a1 dc 52 41 00 81 15 7c 34 40 00 54 41 41 00 c7 45 f8 dc ef

DIS | lift_level=dis
  files=8, total_bytes=1,404,756
  first=d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.asm, size=37,198
  head:
PUSH EBP
MOV EBP,ESP
SUB ESP,0x14
OR EAX,dword ptr [0x0041414c]
MOV EAX,[0x004152dc]

DEC | lift_level=dec
  files=8, total_bytes=12,936,213
  first=d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.c, size=39,109
  head:

/* WARNING: Globals starting with '_' overlap smaller symbols at the same address */

void __fastcall FUN_00401000(undefined4 param_1,uint param_2)



## 5. Train and save one BPE tokenizer per representation

EXE bytes, assembly instructions, and decompiled code have different symbol distributions, so each representation receives its own tokenizer. Each tokenizer learns from at most 4,096 artifacts. Every run rebuilds its vocabulary and merge rules, then writes a fresh `tokenizer.json`.

In [4]:
trained_tokenizers = {}
training_records = {}

for name, config in REPRESENTATIONS.items():
    files = config['files'][:TRAIN_FILE_LIMIT]
    output_dir = TOKENIZER_ROOT / name / 'bpe' / str(VOCAB_SIZE) / str(len(files))
    tokenizer_path = output_dir / 'tokenizer.json'
    print(f'\n=== {name.upper()} BPE ===')
    print(f'files={len(files)}, requested_vocab={VOCAB_SIZE}, output={tokenizer_path}')
    started = time.perf_counter()

    iterator = TokenizationTrainingIterator(
        files, config['lift_level'], batch_size=BATCH_SIZE, block_size=1_024
    ).build()
    print('DECOMPOSITION HEAD')
    for record in iterator.decomposition_records[:5]:
        print(f'  {record["file"][:16]} | {record["bytes"]:,} bytes -> {record["words"]:,} words')
    tokenizer_object = TrainTokenizer(
        iterator, config['lift_level'], TokenizationAlgorithm.BPE,
        VOCAB_SIZE, MAX_TOKEN_LENGTH
    )()
    output_dir.mkdir(parents=True, exist_ok=True)
    tokenizer_object.save(str(tokenizer_path))
    num_characters = iterator.num_characters
    print('WRITE |', tokenizer_path)

    elapsed = time.perf_counter() - started
    realized_vocab = tokenizer_object.get_vocab_size()
    print(f'DONE  | realized_vocab={realized_vocab:,}, elapsed={elapsed:.2f}s')
    trained_tokenizers[name] = tokenizer_object
    training_records[name] = {
        'files': len(files),
        'requested_vocab': VOCAB_SIZE,
        'realized_vocab': realized_vocab,
        'characters': num_characters,
        'elapsed_seconds': elapsed,
        'tokenizer_path': str(tokenizer_path),
        'lift_level': config['lift_level'].value,
    }

pprint(training_records)


=== EXE BPE ===
files=8, requested_vocab=16384, output=/Users/jung/Projects/malrec-lab/malweave/models/tokenizers/exe/bpe/16384/8/tokenizer.json
CACHE | loaded existing tokenizer
DONE  | realized_vocab=16,391, elapsed=0.02s

=== DIS BPE ===
files=8, requested_vocab=16384, output=/Users/jung/Projects/malrec-lab/malweave/models/tokenizers/dis/bpe/16384/8/tokenizer.json
CACHE | loaded existing tokenizer
DONE  | realized_vocab=16,391, elapsed=0.02s

=== DEC BPE ===
files=8, requested_vocab=16384, output=/Users/jung/Projects/malrec-lab/malweave/models/tokenizers/dec/bpe/16384/8/tokenizer.json
CACHE | loaded existing tokenizer
DONE  | realized_vocab=16,391, elapsed=0.02s
{'dec': {'characters': None,
         'elapsed_seconds': 0.017927000000781845,
         'files': 8,
         'lift_level': 'dec',
         'realized_vocab': 16391,
         'requested_vocab': 16384,
         'tokenizer_path': '/Users/jung/Projects/malrec-lab/malweave/models/tokenizers/dec/bpe/16384/8/tokenizer.json'},
 'dis

## 6. Inspect learned vocabularies

The first IDs are the seven special tokens. EXE tokens appear as private Unicode symbols; DIS and DEC tokens contain recognizable assembly instructions and C-like constructs.

In [5]:
for name, tokenizer_object in trained_tokenizers.items():
    vocab_by_id = sorted(tokenizer_object.get_vocab().items(), key=lambda item: item[1])
    print(f'\n{name.upper()} vocabulary head:')
    for token, token_id in vocab_by_id[:20]:
        printable = token if name != 'exe' else token.encode('unicode_escape').decode()
        print(f'  {token_id:>5}: {printable!r}')


EXE vocabulary head:
      0: '<pad>'
      1: '<unk>'
      2: '<msk>'
      3: '<bos>'
      4: '<eos>'
      5: '<cls>'
      6: '<sep>'
      7: '\\u2a00'
      8: '\\u2a01'
      9: '\\u2a02'
     10: '\\u2a03'
     11: '\\u2a04'
     12: '\\u2a05'
     13: '\\u2a06'
     14: '\\u2a07'
     15: '\\u2a08'
     16: '\\u2a09'
     17: '\\u2a0a'
     18: '\\u2a0b'
     19: '\\u2a0c'

DIS vocabulary head:
      0: '<pad>'
      1: '<unk>'
      2: '<msk>'
      3: '<bos>'
      4: '<eos>'
      5: '<cls>'
      6: '<sep>'
      7: ' '
      8: '*'
      9: '+'
     10: ','
     11: '-'
     12: '.'
     13: '0'
     14: '1'
     15: '2'
     16: '3'
     17: '4'
     18: '5'
     19: '6'

DEC vocabulary head:
      0: '<pad>'
      1: '<unk>'
      2: '<msk>'
      3: '<bos>'
      4: '<eos>'
      5: '<cls>'
      6: '<sep>'
      7: ' '
      8: '!'
      9: '"'
     10: '#'
     11: '$'
     12: '%'
     13: '&'
     14: "'"
     15: '('
     16: ')'
     17: '*'
     18: '+'
     

## 7. Apply tokenizers and materialize token IDs

Each notebook-01 artifact is converted to its tokenizer input, encoded, and saved as a JSON array of integer token IDs. These per-sample JSON files are model data, while `manifest.jsonl` is the audit metadata. The log reports sequence length and unknown-token count for every sample.

In [6]:
tokenization_records = {}

for name, config in REPRESENTATIONS.items():
    tokenizer_object = trained_tokenizers[name]
    output_dir = TOKENIZED_ROOT / name / 'bpe' / str(VOCAB_SIZE)
    output_dir.mkdir(parents=True, exist_ok=True)
    unk_id = tokenizer_object.token_to_id(SPECIALS['unk_token'])
    records = []
    print(f'\n=== TOKENIZE {name.upper()} ===')

    for index, path in enumerate(config['files'], start=1):
        content = path.read_bytes()
        text = bytes_to_str_utf8(content) if name == 'exe' else content.decode()
        encoding = tokenizer_object.encode(text, add_special_tokens=False)
        output_path = output_dir / f'{path.stem}.json'
        output_path.write_text(json.dumps(encoding.ids))
        unknowns = encoding.ids.count(unk_id) if unk_id is not None else 0
        record = {
            'sample': path.stem,
            'input_bytes': len(content),
            'tokens': len(encoding.ids),
            'unknown_tokens': unknowns,
            'output': str(output_path),
        }
        records.append(record)
        print(
            f'  [{index}/{len(config["files"])}] {path.stem[:12]} | '
            f'{len(content):,} bytes -> {len(encoding.ids):,} tokens | unk={unknowns}'
        )

    manifest_path = output_dir / 'manifest.jsonl'
    with manifest_path.open('w') as stream:
        for record in records:
            stream.write(json.dumps(record, sort_keys=True) + '\n')
    print('MANIFEST |', manifest_path)
    tokenization_records[name] = records


=== TOKENIZE EXE ===
  [1/8] d00630d78796 | 7,680 bytes -> 3,662 tokens | unk=0
  [2/8] d07092d99a76 | 101,888 bytes -> 43,031 tokens | unk=0
  [3/8] d079e9fcd6bb | 202,240 bytes -> 49,880 tokens | unk=0
  [4/8] d098ebd6d83c | 62,976 bytes -> 27,747 tokens | unk=0
  [5/8] d09ef86191b7 | 2,048 bytes -> 774 tokens | unk=0
  [6/8] d0d2525c3cdd | 237,568 bytes -> 179,447 tokens | unk=0
  [7/8] d0d8504d8c8b | 674,304 bytes -> 495,571 tokens | unk=0
  [8/8] d0d87cb5e049 | 79,872 bytes -> 31,677 tokens | unk=0
MANIFEST | /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/tokenized/exe/bpe/16384/manifest.jsonl

=== TOKENIZE DIS ===
  [1/8] d00630d78796 | 37,198 bytes -> 3,857 tokens | unk=0
  [2/8] d07092d99a76 | 402,890 bytes -> 40,157 tokens | unk=0
  [3/8] d079e9fcd6bb | 331,686 bytes -> 32,666 tokens | unk=0
  [4/8] d098ebd6d83c | 310,374 bytes -> 30,956 tokens | unk=0
  [5/8] d09ef86191b7 | 27 bytes -> 3 tokens | unk=0
  [6/8] d0d2525c3cdd | 27 bytes -> 3 tokens | unk=0
  [7/8

## 8. Inspect tokenized data and sequence-length statistics

This stage previews token IDs and their corresponding learned tokens, then reports minimum, median, mean, and maximum sequence lengths. `bytes_per_token` shows how much source content one token represents on average: a larger value means a shorter encoded sequence.

In [7]:
summary = {}
for name, records in tokenization_records.items():
    lengths = [record['tokens'] for record in records]
    total_bytes = sum(record['input_bytes'] for record in records)
    total_tokens = sum(lengths)
    summary[name] = {
        'samples': len(records),
        'min_tokens': min(lengths),
        'median_tokens': statistics.median(lengths),
        'mean_tokens': round(statistics.mean(lengths), 2),
        'max_tokens': max(lengths),
        'bytes_per_token': round(total_bytes / total_tokens, 3),
        'unknown_tokens': sum(record['unknown_tokens'] for record in records),
    }

    first_ids = json.loads(Path(records[0]['output']).read_text())
    first_tokens = trained_tokenizers[name].id_to_token
    print(f'\n{name.upper()} tokenized head:')
    print('  ids   :', first_ids[:32])
    preview_tokens = [first_tokens(token_id) for token_id in first_ids[:16]]
    if name == 'exe':
        preview_tokens = [token.encode('unicode_escape').decode() for token in preview_tokens]
    print('  tokens:', preview_tokens)

print('\nSequence-length summary:')
pprint(summary)


EXE tokenized head:
  ids   : [3123, 27, 7786, 83, 836, 168, 227, 866, 1659, 131, 835, 91, 836, 13360, 227, 2425, 15307, 226, 2425, 11719, 264, 1054, 24, 4478, 416, 1104, 866, 2535, 79, 416, 63, 267]
  tokens: ['\\u2a55\\u2a8b\\u2aec\\u2a83\\u2aec', '\\u2a14', '\\u2a0b\\u2a05', '\\u2a4c', '\\u2a41\\u2a41\\u2a00', '\\u2aa1', '\\u2adc', '\\u2a52\\u2a41\\u2a00', '\\u2a81\\u2a15', '\\u2a7c', '\\u2a34\\u2a40\\u2a00', '\\u2a54', '\\u2a41\\u2a41\\u2a00', '\\u2ac7\\u2a45\\u2af8', '\\u2adc', '\\u2aef\\u2af2\\u2a0d']

DIS tokenized head:
  ids   : [205, 209, 1733, 7604, 1357, 627, 309, 6731, 1460, 1457, 7047, 85, 102, 245, 2586, 85, 102, 481, 2826, 85, 102, 1103, 85, 102, 888, 1460, 1183, 14010, 309, 9954, 1744, 1183]
  tokens: ['PUSH EBP', 'MOV EBP,ESP', 'SUB ESP,0x14', 'OR EAX,dword', ' ptr [0x004141', '4c]', 'MOV EAX,[0x004', '152dc]', 'ADC dword', ' ptr [0x0040347', 'c],0x414154', 'MOV', ' dword ptr [E', 'BP + -0x8]', ',0xdf2efdc', 'MOV']

DEC tokenized head:
  ids   : [554, 4779, 4785, 477

## 9. Output contract and next step

```text
models/tokenizers/
├── exe/bpe/16384/<num-files>/tokenizer.json
├── dis/bpe/16384/<num-files>/tokenizer.json
└── dec/bpe/16384/<num-files>/tokenizer.json

data/ranDS/processed/tokenized/
├── exe/bpe/16384/<sha256>.json
├── dis/bpe/16384/<sha256>.json
└── dec/bpe/16384/<sha256>.json
```

Each sample JSON contains its complete token-ID sequence; each representation directory also contains `manifest.jsonl`. The next stage is representation digesting and redundancy analysis. For a full-scale experiment, fit each tokenizer only on the intended tokenizer-training subset and preserve its sample manifest with the tokenizer artifact.